# Quantiles and above-median incidence by hazard

For each of the 10 hazard indicators and **within each scenario** (ssp × gcm × time_period), per AGEB:

| Column | Definition |
|---|---|
| `{var}_quantile_nati / _state / _mun` | Percentile rank (0–1, `rank(pct=True)`) among all AGEB in the country / state / municipality |
| `{var}_inciden_median_nati / _state / _mun` | 1 if the value is **≥ the median** of the level, 0 if lower |

- State = `CVEGEO[:2]`, municipality = `CVEGEO[:5]`.
- Values are raw: a high quantile = a high value, with no inversion of the risk direction (e.g. `cold_bio6`, `drought_aridity_annual`). This will be adjusted later.
- Input: `climate_ssp_gcm_all_ageb.parquet` (notebook 01). Output: `climate_ssp_gcm_all_ageb_metrics.parquet`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

def find_hazard_dir(start=Path.cwd().resolve()):
    for p in [start, *start.parents]:
        d = p / "data" / "hazard" / "ageb"
        if (d / "climate_ssp_gcm_all_ageb.parquet").exists():
            return d
    raise FileNotFoundError("Run notebook 01 first")

DIR = find_hazard_dir()
IN_PARQUET = DIR / "climate_ssp_gcm_all_ageb.parquet"
OUT_PARQUET = DIR / "climate_ssp_gcm_all_ageb_metrics.parquet"

climate = pd.read_parquet(IN_PARQUET)
climate["CVEGEO"] = climate["CVEGEO"].astype(str)
SCEN = ["ssp", "gcm", "time_period"]
HAZARDS = [c for c in climate.columns if c not in ["CVEGEO", *SCEN]]
print(len(HAZARDS), "hazards:", HAZARDS)
print(climate.shape)

10 hazards: ['heat_bio2', 'heat_bio5', 'heat_bio10', 'heat_cdd', 'drought_pet_annual', 'drought_aridity_annual', 'cold_bio6', 'cold_bio6_delta', 'flood_bio13', 'flood_bio15']
(19548240, 14)


## 1. Per-scenario function

In [2]:
LEVELS = {"nati": None, "state": "_state", "mun": "_mun"}

def metrics_one_scenario(df):
    """df: AGEB of a single scenario. Returns df with the metric columns for each hazard."""
    df = df.copy()
    keys = {"nati": None, "state": df["CVEGEO"].str[:2], "mun": df["CVEGEO"].str[:5]}
    out = {c: df[c] for c in ["CVEGEO", *SCEN]}
    for h in HAZARDS:
        out[h] = df[h]
        v = df[h]
        q, inc = {}, {}
        for lvl, k in keys.items():
            g = v if k is None else v.groupby(k.values)
            if k is None:
                q[lvl] = v.rank(pct=True, method="average")
                med = v.median()
            else:
                q[lvl] = g.rank(pct=True, method="average")
                med = g.transform("median")
            inc[lvl] = (v >= med).astype("int8")
        for lvl in keys:
            out[f"{h}_quantile_{lvl}"] = q[lvl].astype("float32")
        for lvl in keys:
            out[f"{h}_inciden_median_{lvl}"] = inc[lvl]
    return pd.DataFrame(out)

# quick test on one scenario
test = metrics_one_scenario(climate[(climate.ssp == "ssp_585") & (climate.gcm == "access_cm2") & (climate.time_period == "2021_2040")])
test.filter(like="heat_bio2").describe().round(3)

,heat_bio2,heat_bio2_quantile_nati,heat_bio2_quantile_state,heat_bio2_quantile_mun,heat_bio2_inciden_median_nati,heat_bio2_inciden_median_state,heat_bio2_inciden_median_mun
count,81451.000,81451.000,81451.000,81451.000,81451.0,81451.0,81451.00
mean,15.029,0.500,0.500,0.515,0.5,0.5,0.51
std,2.495,0.289,0.289,0.289,0.5,0.5,0.50
min,3.789,0.000,0.000,0.001,0.0,0.0,0.00
25%,13.425,0.250,0.250,0.264,0.0,0.0,0.00
50%,15.645,0.500,0.500,0.512,1.0,1.0,1.00
75%,16.922,0.750,0.750,0.765,1.0,1.0,1.00
max,20.500,1.000,1.000,1.000,1.0,1.0,1.00


## 2. Apply to the 240 scenarios

In [3]:
parts = []
for i, (_, g) in enumerate(climate.groupby(SCEN, observed=True, sort=True), 1):
    parts.append(metrics_one_scenario(g))
    if i % 40 == 0:
        print(i, "scenarios")
res = pd.concat(parts, ignore_index=True)
del parts
for c in ["ssp", "gcm", "time_period"]:
    res[c] = res[c].astype("category")
print(res.shape)

40 scenarios


80 scenarios


120 scenarios


160 scenarios


200 scenarios


240 scenarios


(19548240, 74)


## 3. Quality checks

In [4]:
qcols = [c for c in res.columns if "_quantile_" in c]
icols = [c for c in res.columns if "_inciden_median_" in c]
assert len(qcols) == len(icols) == 30
assert res[qcols + icols].isna().sum().sum() == 0
assert res[qcols].min().min() > 0 and res[qcols].max().max() <= 1
assert set(np.unique(res[icols].to_numpy()[:100000])) <= {0, 1}

# National: ~50 % of AGEB should be 1 (somewhat more if there are ties at the median)
print("% of AGEB with national incidence = 1 (per hazard):")
print((res[[f"{h}_inciden_median_nati" for h in HAZARDS]].mean() * 100).round(1))

# Municipalities with a single AGEB: quantile = 1 and incidence = 1 (expected edge case)
n_mun = res[res.ssp.eq("ssp_126") & res.gcm.eq("access_cm2") & res.time_period.eq("2021_2040")].CVEGEO.str[:5].value_counts()
print("Municipalities with 1 AGEB:", (n_mun == 1).sum(), "of", len(n_mun))

% of AGEB with national incidence = 1 (per hazard):
heat_bio2_inciden_median_nati                 50.0
heat_bio5_inciden_median_nati                 50.0
heat_bio10_inciden_median_nati                50.0
heat_cdd_inciden_median_nati                  50.0
drought_pet_annual_inciden_median_nati        50.0
drought_aridity_annual_inciden_median_nati    50.0
cold_bio6_inciden_median_nati                 50.0
cold_bio6_delta_inciden_median_nati           59.7
flood_bio13_inciden_median_nati               50.0
flood_bio15_inciden_median_nati               50.0
dtype: float64
Municipalities with 1 AGEB: 0 of 2469


## 4. Export

In [5]:
res = res.sort_values(["CVEGEO", *SCEN]).reset_index(drop=True)
res.to_parquet(OUT_PARQUET, index=False, compression="zstd")
print(OUT_PARQUET, f"{OUT_PARQUET.stat().st_size/1e6:.0f} MB")
res.head()

/Users/fabianfuentes/git/adatpation_ageb_data/data/hazard/ageb/climate_ssp_gcm_all_ageb_metrics.parquet 1339 MB


,CVEGEO,ssp,gcm,time_period,heat_bio2,heat_bio2_quantile_nati,heat_bio2_quantile_state,heat_bio2_quantile_mun,heat_bio2_inciden_median_nati,heat_bio2_inciden_median_state,...,flood_bio13_inciden_median_nati,flood_bio13_inciden_median_state,flood_bio13_inciden_median_mun,flood_bio15,flood_bio15_quantile_nati,flood_bio15_quantile_state,flood_bio15_quantile_mun,flood_bio15_inciden_median_nati,flood_bio15_inciden_median_state,flood_bio15_inciden_median_mun
0,0100100010017,ssp_126,access_cm2,2021_2040,16.864100,0.726105,0.468354,0.586895,1,0,...,0,1,0,110.297997,0.738972,0.251356,0.031339,1,0,0
1,0100100010017,ssp_126,access_cm2,2041_2060,16.972401,0.720132,0.470163,0.586895,1,0,...,0,1,0,108.491997,0.682901,0.249548,0.022792,1,0,0
2,0100100010017,ssp_126,access_cm2,2061_2080,16.989100,0.725964,0.475588,0.595442,1,0,...,0,1,0,110.857002,0.702834,0.249548,0.022792,1,0,0
3,0100100010017,ssp_126,access_cm2,2081_2100,17.114100,0.734810,0.470163,0.586895,1,0,...,0,1,0,109.959000,0.688727,0.249548,0.022792,1,0,0
4,0100100010017,ssp_126,bcc_csm2_mr,2021_2040,17.839100,0.785644,0.470163,0.586895,1,0,...,0,1,0,109.978996,0.681060,0.253165,0.025641,1,0,0
